# Olist KPI Analysis

## 목적

Olist의 핵심 비즈니스 지표(KPI)를 계산하여 매출, 주문, 고객, 상품, 배송 성과를 분석한다.

이 결과는 Tableau Dashboard 구축의 기반 데이터로 활용한다.

### KPI

- Total Revenue
- Total Orders
- Total Customers
- Average Order Value (AOV)
- Category Sales
- Regional Sales
- Delivery Lead Time

In [4]:
from pathlib import Path
import sqlite3
import pandas as pd

BASE_DIR = Path.cwd().parent
DB_PATH = BASE_DIR / "database" / "olist_dashboard.db"

conn = sqlite3.connect(DB_PATH)

## 1. Total Product Sales

전체 상품 판매금액을 확인한다.

상품 판매금액은 `order_items.price`의 합계로 정의하며, 배송비는 제외한다.

In [6]:
revenue = pd.read_sql("""
SELECT SUM(price) AS total_product_sales
FROM order_items;
""", conn)

revenue

,total_product_sales
0,13591643.7


In [7]:
print(f"Total Product Sales: ${revenue.loc[0, 'total_product_sales']:,.2f}")

Total Product Sales: $13,591,643.70


### 결과

- 전체 상품 판매금액은 **$13,591,643.70**으로 확인되었다.
- 본 지표는 상품 가격만 포함하며 배송비는 제외한다.
- 이후 월별 판매금액 추이와 카테고리별 성과 분석의 기준 지표로 활용한다.

In [9]:
sales_by_status = pd.read_sql("""
SELECT o.order_status, SUM(oi.price) AS product_sales
FROM orders AS o
JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY o.order_status
ORDER BY product_sales DESC;
""", conn)

sales_by_status

,order_status,product_sales
0,delivered,13221498.11
1,shipped,150727.44
2,canceled,95235.27
3,invoiced,61526.37
4,processing,60439.22
5,unavailable,2007.69
6,approved,209.60


In [10]:
delivered_sales = pd.read_sql("""
SELECT SUM(oi.price) AS delivered_product_sales
FROM orders AS o
JOIN order_items AS oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered';
""", conn)

print(f"Delivered Product Sales: ${delivered_sales.loc[0, 'delivered_product_sales']:,.2f}")

Delivered Product Sales: $13,221,498.11


In [11]:
non_delivered_sales = (
    revenue.loc[0, "total_product_sales"] - delivered_sales.loc[0, "delivered_product_sales"]
)

non_delivered_ratio = (
    non_delivered_sales / revenue.loc[0, "total_product_sales"]
)

print(f"Non-delivered Sales: ${non_delivered_sales:,.2f}")
print(f"Non-delivered Ratio: {non_delivered_ratio:.2%}")

Non-delivered Sales: $370,145.59
Non-delivered Ratio: 2.72%


### KPI Selection

대표 매출 KPI를 선정하기 위해 전체 상품 판매금액과 배송 완료 상품 판매금액을 비교하였다.

배송 완료되지 않은 주문의 판매금액은 전체의 2.72%로 확인되었다.

따라서 이후 분석 및 Tableau Dashboard에서는 실제 거래가 완료된 **Delivered Product Sales**를 대표 매출 지표로 사용한다.

## 2. Total Orders

전체 주문 수를 확인한다.

주문 수는 `orders` 테이블의 `order_id`를 기준으로 계산한다.

In [14]:
orders = pd.read_sql("""
SELECT COUNT(order_id) AS total_orders
FROM orders;
""", conn)

orders

,total_orders
0,99441


In [15]:
print(f"Total Orders: {orders.loc[0, 'total_orders']:,}")

Total Orders: 99,441


orders_by_status = pd.read_sql("""
SELECT order_status, COUNT(order_id) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
""", conn)

orders_by_status

In [17]:
delivered_orders = pd.read_sql("""
SELECT COUNT(order_id) AS delivered_orders
FROM orders
WHERE order_status = 'delivered';
""", conn)

delivered_orders

,delivered_orders
0,96478


In [18]:
print(f"Delivered Orders: {delivered_orders.loc[0, 'delivered_orders']:,}")

Delivered Orders: 96,478


In [19]:
delivered_order_ratio = (
    delivered_orders.loc[0, "delivered_orders"] / orders.loc[0, "total_orders"]
)

print(f"Delivered Order Ratio: {delivered_order_ratio:.2%}")

Delivered Order Ratio: 97.02%


### KPI Selection

전체 주문과 배송 완료 주문을 비교하여 대표 주문 KPI를 검토하였다.

배송 완료 주문은 전체 주문의 97.02%를 차지하였다.

따라서 본 프로젝트에서는 실제 완료된 거래를 기준으로 **Delivered Orders**를 핵심 주문 KPI로 활용한다.

## 3. Total Customers

전체 고객 수를 확인한다.

고객 수는 `customers` 테이블을 기준으로 계산하며,
`customer_id`와 `customer_unique_id`의 차이를 함께 검토한다.

In [22]:
customers = pd.read_sql("""
SELECT COUNT(customer_id) AS total_customers
FROM customers;
""", conn)

customers

,total_customers
0,99441


In [23]:
print(f"Total Customers: {customers.loc[0, 'total_customers']:,}")

Total Customers: 99,441


In [24]:
unique_customers = pd.read_sql("""
SELECT COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM customers;
""", conn)

unique_customers

,unique_customers
0,96096


In [25]:
print(f"Unique Customers: {unique_customers.loc[0, 'unique_customers']:,}")

Unique Customers: 96,096


In [26]:
duplicate_customer_ids = (
    customers.loc[0, "total_customers"] - unique_customers.loc[0, "unique_customers"]
)

duplicate_customer_ratio = (
    duplicate_customer_ids / customers.loc[0, "total_customers"]
)

print(f"Duplicate Customer IDs: {duplicate_customer_ids:,}")
print(f"Duplicate Ratio: {duplicate_customer_ratio:.2%}")

Duplicate Customer IDs: 3,345
Duplicate Ratio: 3.36%


### KPI Selection

`customer_id`와 `customer_unique_id`를 비교하여 대표 고객 KPI를 검토하였다.

분석 결과, 동일 고객이 여러 `customer_id`를 가질 수 있는 것으로 확인되었다.

따라서 실제 고객 수를 정확하게 반영하는 `customer_unique_id`를 대표 고객 KPI로 활용하였다.

In [28]:
same_customer = pd.read_sql("""
SELECT
    customer_unique_id,
    COUNT(customer_id) AS customer_count
FROM customers
GROUP BY customer_unique_id
HAVING COUNT(customer_id) > 1
ORDER BY customer_count DESC;
""", conn)

same_customer.head(10)

,customer_unique_id,customer_count
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6


In [29]:
print(f"Customers with multiple customer_id: {len(same_customer):,}")

Customers with multiple customer_id: 2,997


### Validation

`customer_unique_id`를 기준으로 검증한 결과, **2,997명의 고객이 2개 이상의 `customer_id`를 보유**하고 있는 것으로 확인되었다.

이로 인해 총 **3,345개의 중복 `customer_id`**가 발생하였으며, 실제 고객 수를 집계할 때는 `customer_unique_id`를 사용하는 것이 적절하다고 판단하였다.

## 4. Average Order Value (AOV)

평균 주문금액(AOV)을 확인한다.

대표 KPI로 선정한 **Delivered Product Sales**와 **Delivered Orders**를 기준으로 계산한다.

### Overall AOV

대표 KPI로 선정한 **Delivered Product Sales**와 **Delivered Orders**를 이용하여 전체 평균 주문금액(AOV)을 계산한다.

이 값은 Executive Dashboard의 핵심 KPI로 활용한다.

In [33]:
aov = (
    delivered_sales.loc[0, "delivered_product_sales"] / delivered_orders.loc[0, "delivered_orders"]
)

print(f"Average Order Value (AOV): ${aov:,.2f}")

Average Order Value (AOV): $137.04


### Monthly AOV

배송 완료 주문을 기준으로 월별 평균 주문금액(AOV)을 계산한다.

월별 AOV 변화를 통해 고객의 구매 규모와 판매 패턴의 변화를 확인한다.

In [35]:
monthly_aov = pd.read_sql("""
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
    SUM(oi.price) AS product_sales,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders AS o
JOIN order_items AS oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY order_month
ORDER BY order_month;
""", conn)

monthly_aov.head()

,order_month,product_sales,total_orders
0,2016-09,134.97,1
1,2016-10,40325.11,265
2,2016-12,10.90,1
3,2017-01,111798.36,750
4,2017-02,234223.40,1653


In [36]:
monthly_aov["aov"] = (
    monthly_aov["product_sales"] / monthly_aov["total_orders"]
)

monthly_aov.head()

,order_month,product_sales,total_orders,aov
0,2016-09,134.97,1,134.970000
1,2016-10,40325.11,265,152.170226
2,2016-12,10.90,1,10.900000
3,2017-01,111798.36,750,149.064480
4,2017-02,234223.40,1653,141.695947


### Highest / Lowest Month

월별 AOV를 비교하여 평균 주문금액이 가장 높고 낮은 기간을 확인한다.

단, 주문 수가 매우 적은 월은 평균값의 대표성이 낮을 수 있으므로, 배송 완료 주문이 100건 이상인 월을 기준으로 비교하였다.

In [38]:
monthly_aov.sort_values("aov", ascending=False).head()

,order_month,product_sales,total_orders,aov
1,2016-10,40325.11,265,152.170226
3,2017-01,111798.36,750,149.064480
6,2017-04,340669.68,2303,147.924307
11,2017-09,607399.67,4150,146.361366
19,2018-05,977544.69,6749,144.842894


In [39]:
monthly_aov.sort_values("aov").head()

,order_month,product_sales,total_orders,aov
2,2016-12,10.90,1,10.900000
9,2017-07,481604.52,3872,124.381333
16,2018-02,826437.13,6555,126.077365
15,2018-01,924645.00,7069,130.802801
14,2017-12,726033.19,5513,131.694756


2016-12는 배송 완료 주문이 1건에 불과하여 대표성이 낮다.

따라서 월별 AOV 비교에서는 주문 수가 충분한 기간을 중심으로 해석한다

In [41]:
monthly_aov_filtered = monthly_aov[
    monthly_aov["total_orders"] >= 100
]

monthly_aov_filtered.sort_values("aov").head()

,order_month,product_sales,total_orders,aov
9,2017-07,481604.52,3872,124.381333
16,2018-02,826437.13,6555,126.077365
15,2018-01,924645.00,7069,130.802801
14,2017-12,726033.19,5513,131.694756
22,2018-08,838576.64,6351,132.038520


In [42]:
highest_aov = monthly_aov_filtered.loc[
    monthly_aov_filtered["aov"].idxmax()
]

lowest_aov = monthly_aov_filtered.loc[
    monthly_aov_filtered["aov"].idxmin()
]

print(
    f"Highest AOV: {highest_aov['order_month']} "
    f"(${highest_aov['aov']:,.2f})"
)

print(
    f"Lowest AOV: {lowest_aov['order_month']} "
    f"(${lowest_aov['aov']:,.2f})"
)

print(
    f"AOV Gap: ${highest_aov['aov'] - lowest_aov['aov']:,.2f}"
)

Highest AOV: 2016-10 ($152.17)
Lowest AOV: 2017-07 ($124.38)
AOV Gap: $27.79


### Business Insight

월별 AOV를 비교한 결과, 평균 주문금액은 월별로 차이를 보였다.

2016-10은 AOV가 152.17로 가장 높았고, 2017-07은 124.38로 가장 낮게 나타났다. 초기 데이터인 2016-09와 2016-12는 주문 수가 매우 적어 비교 대상에서 제외하였다.

월별 AOV 변동의 원인은 본 분석만으로는 확인할 수 없으며, 판매 카테고리와 주문 구성 변화를 함께 분석하면 보다 구체적인 해석이 가능할 것이다.

## 5. Category Performance

상품 카테고리별 판매 성과를 분석한다.

매출과 주문 수를 함께 확인하여 어떤 카테고리가 비즈니스 성과를 주도했는지 파악한다.

In [45]:
category_sales = pd.read_sql("""
SELECT p.product_category_name, SUM(oi.price) AS product_sales
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
JOIN orders AS o
    ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name
ORDER BY product_sales DESC;
""", conn)

category_sales.head(10)

,product_category_name,product_sales
0,beleza_saude,1233131.72
1,relogios_presentes,1166176.98
2,cama_mesa_banho,1023434.76
3,esporte_lazer,954852.55
4,informatica_acessorios,888724.61
5,moveis_decoracao,711927.69
6,utilidades_domesticas,615628.69
7,cool_stuff,610204.10
8,automotivo,578966.65
9,brinquedos,471286.48


### Sales Share

카테고리별 매출이 전체 매출에서 차지하는 비중을 확인한다.

매출 규모뿐 아니라 전체 비즈니스에 미치는 영향력을 함께 분석한다.

In [47]:
category_sales["sales_share"] = (
    category_sales["product_sales"]
    / category_sales["product_sales"].sum()
)

category_sales["sales_share"] = (
    category_sales["sales_share"] * 100
).round(2)

category_sales.head(10)

,product_category_name,product_sales,sales_share
0,beleza_saude,1233131.72,9.33
1,relogios_presentes,1166176.98,8.82
2,cama_mesa_banho,1023434.76,7.74
3,esporte_lazer,954852.55,7.22
4,informatica_acessorios,888724.61,6.72
5,moveis_decoracao,711927.69,5.38
6,utilidades_domesticas,615628.69,4.66
7,cool_stuff,610204.10,4.62
8,automotivo,578966.65,4.38
9,brinquedos,471286.48,3.56


### Category Order Count

카테고리별 배송 완료 주문 수를 확인한다.

카테고리 매출이 주문량에 의해 발생한 것인지, 주문당 판매금액이 높은 결과인지 분석하기 위한 기준으로 활용한다.

In [49]:
category_orders = pd.read_sql("""
SELECT
    p.product_category_name,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
JOIN orders AS o
    ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name
ORDER BY total_orders DESC;
""", conn)

category_orders.head(10)

,product_category_name,total_orders
0,cama_mesa_banho,9272
1,beleza_saude,8647
2,esporte_lazer,7530
3,informatica_acessorios,6530
4,moveis_decoracao,6307
5,utilidades_domesticas,5743
6,relogios_presentes,5495
7,telefonia,4093
8,automotivo,3810
9,brinquedos,3804


### Category Average Order Value (AOV)

카테고리별 평균 주문금액(AOV)을 계산한다.

카테고리의 매출 규모가 주문량 때문인지, 주문당 구매금액 때문인지 함께 분석한다.

In [51]:
category_aov = category_sales.merge(
    category_orders,
    on="product_category_name"
)

category_aov.head()

,product_category_name,product_sales,sales_share,total_orders
0,beleza_saude,1233131.72,9.33,8647
1,relogios_presentes,1166176.98,8.82,5495
2,cama_mesa_banho,1023434.76,7.74,9272
3,esporte_lazer,954852.55,7.22,7530
4,informatica_acessorios,888724.61,6.72,6530


In [52]:
category_aov["aov"] = (
    category_aov["product_sales"] / category_aov["total_orders"]
).round(2)

category_aov.head(10)

,product_category_name,product_sales,sales_share,total_orders,aov
0,beleza_saude,1233131.72,9.33,8647,142.61
1,relogios_presentes,1166176.98,8.82,5495,212.23
2,cama_mesa_banho,1023434.76,7.74,9272,110.38
3,esporte_lazer,954852.55,7.22,7530,126.81
4,informatica_acessorios,888724.61,6.72,6530,136.10
5,moveis_decoracao,711927.69,5.38,6307,112.88
6,utilidades_domesticas,615628.69,4.66,5743,107.20
7,cool_stuff,610204.10,4.62,3559,171.45
8,automotivo,578966.65,4.38,3810,151.96
9,brinquedos,471286.48,3.56,3804,123.89


In [53]:
category_aov.sort_values("aov", ascending=False).head(5)

,product_category_name,product_sales,sales_share,total_orders,aov
16,pcs,218684.14,1.65,177,1235.50
35,portateis_casa_forno_e_cafe,46589.56,0.35,72,647.08
26,eletrodomesticos_2,107953.95,0.82,227,475.57
29,agro_industria_e_comercio,70566.10,0.53,177,398.68
62,portateis_cozinha_e_preparadores_de_alimentos,3933.63,0.03,13,302.59


In [54]:
category_aov.sort_values("aov").head(5)

,product_category_name,product_sales,sales_share,total_orders,aov
70,casa_conforto_2,760.27,0.01,24,31.68
69,flores,1110.04,0.01,29,38.28
67,fraldas_higiene,1500.79,0.01,25,60.03
71,cds_dvds_musicais,730.00,0.01,12,60.83
21,eletronicos,155043.93,1.17,2517,61.60


### Top / Bottom Category

매출, 주문 수, 평균 주문금액(AOV)을 함께 고려하여 주요 카테고리의 특성을 확인한다.

In [56]:
category_aov.sort_values("aov",ascending=False).head(10)

,product_category_name,product_sales,sales_share,total_orders,aov
16,pcs,218684.14,1.65,177,1235.50
35,portateis_casa_forno_e_cafe,46589.56,0.35,72,647.08
26,eletrodomesticos_2,107953.95,0.82,227,475.57
29,agro_industria_e_comercio,70566.10,0.53,177,398.68
62,portateis_cozinha_e_preparadores_de_alimentos,3933.63,0.03,13,302.59
18,instrumentos_musicais,184315.74,1.39,611,301.66
19,eletroportateis,182754.12,1.38,609,300.09
32,telefonia_fixa,55315.21,0.42,212,260.92
39,construcao_ferramentas_seguranca,38773.22,0.29,159,243.86
33,climatizacao,53323.56,0.40,246,216.76


In [57]:
category_aov.sort_values("aov").head(10)

,product_category_name,product_sales,sales_share,total_orders,aov
70,casa_conforto_2,760.27,0.01,24,31.68
69,flores,1110.04,0.01,29,38.28
67,fraldas_higiene,1500.79,0.01,25,60.03
71,cds_dvds_musicais,730.00,0.01,12,60.83
21,eletronicos,155043.93,1.17,2517,61.60
41,alimentos,28731.15,0.22,441,65.15
51,alimentos_bebidas,14942.88,0.11,221,67.61
54,artigos_de_natal,8737.84,0.07,125,69.90
49,livros_tecnicos,18702.23,0.14,256,73.06
63,fashion_roupa_feminina,2634.94,0.02,36,73.19


주문 수가 매우 적은 카테고리는 평균 주문금액(AOV)이 크게 왜곡될 수 있다.

따라서 대표성이 높은 카테고리를 확인하기 위해 주문 수 기준을 적용하여 추가 분석을 수행한다.

In [59]:
category_aov_filtered = category_aov[
    category_aov["total_orders"] >= 100
]

category_aov_filtered.sort_values("aov",ascending=False).head(10)

,product_category_name,product_sales,sales_share,total_orders,aov
16,pcs,218684.14,1.65,177,1235.50
26,eletrodomesticos_2,107953.95,0.82,227,475.57
29,agro_industria_e_comercio,70566.10,0.53,177,398.68
18,instrumentos_musicais,184315.74,1.39,611,301.66
19,eletroportateis,182754.12,1.38,609,300.09
32,telefonia_fixa,55315.21,0.42,212,260.92
39,construcao_ferramentas_seguranca,38773.22,0.29,159,243.86
33,climatizacao,53323.56,0.40,246,216.76
14,moveis_escritorio,268154.31,2.03,1254,213.84
1,relogios_presentes,1166176.98,8.82,5495,212.23


In [60]:
category_aov_filtered.sort_values(
    "aov"
).head(10)

,product_category_name,product_sales,sales_share,total_orders,aov
21,eletronicos,155043.93,1.17,2517,61.60
41,alimentos,28731.15,0.22,441,65.15
51,alimentos_bebidas,14942.88,0.11,221,67.61
54,artigos_de_natal,8737.84,0.07,125,69.90
49,livros_tecnicos,18702.23,0.14,256,73.06
46,bebidas,21529.84,0.16,287,75.02
13,telefonia,309860.23,2.34,4093,75.70
53,fashion_underwear_e_moda_praia,9305.95,0.07,117,79.54
22,fashion_bolsas_e_acessorios,149329.39,1.13,1820,82.05
37,livros_interesse_geral,45302.15,0.34,496,91.33


### Business Insight

카테고리별 매출, 주문 수, 평균 주문금액(AOV)을 비교한 결과, 판매 구조가 카테고리마다 뚜렷하게 달랐다.

`pcs`는 주문 수가 177건으로 많지 않았지만 AOV는 1,235.50으로 가장 높아 고가 상품 중심의 판매 특성을 보였다.

반면 `relogios_presentes`는 5,495건의 주문과 AOV 212.23을 기록하며 높은 주문량을 기반으로 안정적인 매출을 창출했다.

즉, 같은 매출이라도 카테고리에 따라 '고객당 구매금액'과 '주문량'의 기여 방식이 서로 다르게 나타났다.

# 6. Regional Performance

지역별 판매 성과를 분석한다.

매출, 주문 수, 고객 수를 비교하여 지역별 비즈니스 성과를 파악한다.

## Sales by State

지역(State)별 배송 완료 매출을 확인한다.

지역별 매출 규모를 비교하여 주요 시장과 매출이 집중된 지역을 파악한다.

In [64]:
state_sales = pd.read_sql("""
SELECT
    c.customer_state,
    SUM(oi.price) AS product_sales
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
JOIN order_items AS oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY product_sales DESC;
""", conn)

state_sales.head(10)

,customer_state,product_sales
0,SP,5067633.16
1,RJ,1759651.13
2,MG,1552481.83
3,RS,728897.47
4,PR,666063.51
5,SC,507012.13
6,BA,493584.14
7,DF,296498.41
8,GO,282836.70
9,ES,268643.45


### Sales Share by State

지역별 매출이 전체 매출에서 차지하는 비중을 확인한다.

매출 규모뿐 아니라 지역별 비즈니스 영향력을 함께 분석한다.

In [66]:
state_sales["sales_share"] = (
    state_sales["product_sales"]
    / state_sales["product_sales"].sum()
    * 100
).round(2)

state_sales.head(10)

,customer_state,product_sales,sales_share
0,SP,5067633.16,38.33
1,RJ,1759651.13,13.31
2,MG,1552481.83,11.74
3,RS,728897.47,5.51
4,PR,666063.51,5.04
5,SC,507012.13,3.83
6,BA,493584.14,3.73
7,DF,296498.41,2.24
8,GO,282836.70,2.14
9,ES,268643.45,2.03


### Orders by State

지역(State)별 배송 완료 주문 수를 확인한다.

매출과 주문 수를 함께 비교하여 지역별 판매 규모를 분석한다.

In [68]:
state_orders = pd.read_sql("""
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_orders DESC;
""", conn)

state_orders.head(10)

,customer_state,total_orders
0,SP,40501
1,RJ,12350
2,MG,11354
3,RS,5345
4,PR,4923
5,SC,3546
6,BA,3256
7,DF,2080
8,ES,1995
9,GO,1957


## Regional Performance Table

매출, 매출 비중, 주문 수를 하나의 테이블로 통합하여 지역별 성과를 비교한다.

In [70]:
state_performance = (
    state_sales
    .merge(state_orders, on="customer_state")
)

state_performance.head()

,customer_state,product_sales,sales_share,total_orders
0,SP,5067633.16,38.33,40501
1,RJ,1759651.13,13.31,12350
2,MG,1552481.83,11.74,11354
3,RS,728897.47,5.51,5345
4,PR,666063.51,5.04,4923


### Customers by State

지역(State)별 고객 수를 확인한다.

주문 수와 고객 수를 함께 비교하여 지역별 고객 규모를 분석한다.

In [72]:
state_customers = pd.read_sql("""
SELECT
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS total_customers
FROM customers
GROUP BY customer_state
ORDER BY total_customers DESC;
""", conn)

state_customers.head(10)

,customer_state,total_customers
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882
5,SC,3534
6,BA,3277
7,DF,2075
8,ES,1964
9,GO,1952


## Update Regional Performance

고객 수를 추가하여 지역별 성과 테이블을 확장한다.

In [74]:
state_performance = state_performance.merge(
    state_customers,
    on="customer_state"
)

state_performance.head()

,customer_state,product_sales,sales_share,total_orders,total_customers
0,SP,5067633.16,38.33,40501,40302
1,RJ,1759651.13,13.31,12350,12384
2,MG,1552481.83,11.74,11354,11259
3,RS,728897.47,5.51,5345,5277
4,PR,666063.51,5.04,4923,4882


### Sales per Customer

지역(State)별 고객 1명당 평균 판매금액을 계산한다.

지역별 전체 매출을 고유 고객 수로 나누어, 지역 규모와 별개로 고객당 구매 규모를 비교한다.

In [76]:
state_performance["sales_per_customer"] = (
    state_performance["product_sales"]
    / state_performance["total_customers"]
).round(2)

state_performance.head(15)

,customer_state,product_sales,sales_share,total_orders,total_customers,sales_per_customer
0,SP,5067633.16,38.33,40501,40302,125.74
1,RJ,1759651.13,13.31,12350,12384,142.09
2,MG,1552481.83,11.74,11354,11259,137.89
3,RS,728897.47,5.51,5345,5277,138.13
4,PR,666063.51,5.04,4923,4882,136.43
5,SC,507012.13,3.83,3546,3534,143.47
6,BA,493584.14,3.73,3256,3277,150.62
7,DF,296498.41,2.24,2080,2075,142.89
8,GO,282836.70,2.14,1957,1952,144.90
9,ES,268643.45,2.03,1995,1964,136.78


### Filter Small Markets

고객 수가 너무 적은 지역은 고객당 평균 판매금액이 크게 변동할 수 있다.

대표성을 높이기 위해 일정 규모 이상의 고객을 보유한 지역만 비교한다.

In [78]:
state_performance_filtered = state_performance[
    state_performance["total_customers"] >= 1000
]

state_performance_filtered.head()

,customer_state,product_sales,sales_share,total_orders,total_customers,sales_per_customer
0,SP,5067633.16,38.33,40501,40302,125.74
1,RJ,1759651.13,13.31,12350,12384,142.09
2,MG,1552481.83,11.74,11354,11259,137.89
3,RS,728897.47,5.51,5345,5277,138.13
4,PR,666063.51,5.04,4923,4882,136.43


### Highest / Lowest State



In [80]:
highest_state = state_performance_filtered.loc[
    state_performance_filtered["sales_per_customer"].idxmax()
]

lowest_state = state_performance_filtered.loc[
    state_performance_filtered["sales_per_customer"].idxmin()
]

In [81]:
print(f"Highest Sales per Customer: {highest_state['customer_state']} ({highest_state['sales_per_customer']:.2f})")

print(f"Lowest Sales per Customer: {lowest_state['customer_state']} ({lowest_state['sales_per_customer']:.2f})")

Highest Sales per Customer: CE (167.37)
Lowest Sales per Customer: SP (125.74)


### Business Insight

고객 수 1,000명 이상 지역을 비교한 결과, 고객당 평균 판매금액은 지역마다 차이를 보였다.

SP는 전체 매출과 고객 수 모두 1위를 기록했지만, 고객당 평균 판매금액은 125.74로 최고 수준은 아니었다.

반면 CE는 시장 규모는 상대적으로 작지만, 고객당 평균 판매금액은 167.37로 가장 높게 나타났다.

즉, 시장 규모가 큰 지역이 항상 고객당 구매 규모까지 높은 것은 아니며, 두 지표를 함께 살펴볼 필요가 있음을 확인했다.

# 07. Delivery Performance

배송 성과는 고객 만족도와 운영 효율을 평가하는 핵심 지표이다.

본 섹션에서는 다음 배송 KPI를 중심으로 물류 성과를 분석한다.

- **Average Delivery Time:** 주문 구매일부터 고객 배송 완료일까지의 평균 배송 기간
- **Delivery Time by State:** 지역별 평균 배송 기간
- **On-time Delivery Rate:** 예상 배송일 대비 정시 배송 비율

## Average Delivery Time

배송 완료(`delivered`) 주문을 대상으로 한다.

주문 구매일(`order_purchase_timestamp`)부터 고객 배송 완료일(`order_delivered_customer_date`)까지의 평균 배송 기간을 계산하여 전체 배송 성과를 확인한다.

In [85]:
delivery_time = pd.read_sql("""
SELECT
    AVG(
        julianday(o.order_delivered_customer_date)
        - julianday(o.order_purchase_timestamp)
    ) AS avg_delivery_days
FROM orders AS o
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL;
""", conn)

delivery_time

,avg_delivery_days
0,12.558217


In [86]:
print(f"Average Delivery Time: {delivery_time.loc[0, 'avg_delivery_days']:.2f} days")

Average Delivery Time: 12.56 days


### Business Insight

배송 완료 주문의 평균 배송 기간은 12.56일이었다.

평균 배송 기간은 전체 배송 성과를 보여주는 대표 지표이지만, 지역별 배송 환경에 따라 차이가 발생할 수 있다.

따라서 다음 단계에서는 지역별 배송 기간과 예상 배송일 대비 정시 배송률을 함께 비교하여 배송 성과를 구체적으로 분석한다.

## Delivery Time by State

배송 완료(`delivered`) 주문을 대상으로 지역(State)별 평균 배송 기간을 계산한다.

지역별 배송 속도를 비교하여 배송이 상대적으로 오래 걸리는 지역과 빠른 지역을 확인한다.

In [89]:
delivery_by_state = pd.read_sql("""
SELECT
    c.customer_state,
    AVG(
        julianday(o.order_delivered_customer_date)
        - julianday(o.order_purchase_timestamp)
    ) AS avg_delivery_days
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC;
""", conn)

delivery_by_state.head(10)

,customer_state,avg_delivery_days
0,RR,29.387546
1,AP,27.185068
2,AM,26.425991
3,AL,24.543855
4,PA,23.772917
5,MA,21.572976
6,SE,21.519788
7,CE,21.266579
8,AC,21.035713
9,PB,20.426768


In [90]:
highest_delivery = delivery_by_state.loc[
    delivery_by_state["avg_delivery_days"].idxmax()
]

lowest_delivery = delivery_by_state.loc[
    delivery_by_state["avg_delivery_days"].idxmin()
]

print(f"Highest Delivery Time: {highest_delivery['customer_state']} ({highest_delivery['avg_delivery_days']:.2f} days)")
print(f"Lowest Delivery Time: {lowest_delivery['customer_state']} ({lowest_delivery['avg_delivery_days']:.2f} days)")

Highest Delivery Time: RR (29.39 days)
Lowest Delivery Time: SP (8.76 days)


### Delivery Count by State

지역(State)별 배송 완료 주문 수를 확인한다.

평균 배송 기간과 함께 비교하여 대표성이 있는 지역을 중심으로 배송 성과를 분석한다.

In [92]:
delivery_count = pd.read_sql("""
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_deliveries
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_deliveries DESC;
""", conn)

delivery_count.head(10)

,customer_state,total_deliveries
0,SP,40501
1,RJ,12350
2,MG,11354
3,RS,5345
4,PR,4923
5,SC,3546
6,BA,3256
7,DF,2080
8,ES,1995
9,GO,1957


In [93]:
delivery_performance = delivery_by_state.merge(
    delivery_count,
    on="customer_state"
)

delivery_performance.head()

,customer_state,avg_delivery_days,total_deliveries
0,RR,29.387546,41
1,AP,27.185068,67
2,AM,26.425991,145
3,AL,24.543855,397
4,PA,23.772917,946


### Filter Small Markets

배송 건수가 적은 지역은 일부 주문의 영향으로 평균 배송 기간이 크게 달라질 수 있다.

따라서 배송 완료 주문이 1,000건 이상인 지역만 대상으로 비교한다.

In [95]:
delivery_performance_filtered = delivery_performance[
    delivery_performance["total_deliveries"] >= 1000
]

delivery_performance_filtered.head()

,customer_state,avg_delivery_days,total_deliveries
7,CE,21.266579,1279
12,BA,19.335466,3256
14,PE,18.448323,1593
17,ES,15.789307,1995
19,GO,15.606339,1957


In [96]:
highest_delivery_filtered = delivery_performance_filtered.loc[
    delivery_performance_filtered["avg_delivery_days"].idxmax()
]

lowest_delivery_filtered = delivery_performance_filtered.loc[
    delivery_performance_filtered["avg_delivery_days"].idxmin()
]

print(f"Highest Delivery Time: {highest_delivery_filtered['customer_state']} ({highest_delivery_filtered['avg_delivery_days']:.2f} days)")
print(f"Lowest Delivery Time: {lowest_delivery_filtered['customer_state']} ({lowest_delivery_filtered['avg_delivery_days']:.2f} days)")

Highest Delivery Time: CE (21.27 days)
Lowest Delivery Time: SP (8.76 days)


### Business Insight

배송 완료 주문이 1,000건 이상인 지역을 비교한 결과, 평균 배송 기간은 지역별로 큰 차이를 보였다.

가장 긴 지역은 CE(21.27일), 가장 짧은 지역은 SP(8.76일)로 약 12.5일의 차이가 확인되었다.

이는 동일한 서비스에서도 지역에 따라 배송 성과가 달라질 수 있음을 보여주며, 다음 단계에서는 예상 배송일 대비 정시 배송률을 함께 비교하여 배송 성과를 추가로 분석한다.

## On-time Delivery Rate

배송 완료(delivered) 주문을 대상으로 예상 배송일과 실제 배송 완료일을 비교한다.

예상 배송일 이전(또는 동일한 날짜)에 배송된 주문의 비율을 계산하여 배송 약속 준수율을 확인한다.

In [99]:
on_time_delivery = pd.read_sql("""
SELECT
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN order_delivered_customer_date <= order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS on_time_deliveries
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;
""", conn)

on_time_delivery

,total_deliveries,on_time_deliveries
0,96470,88644


In [100]:
on_time_delivery["on_time_rate"] = (
    on_time_delivery["on_time_deliveries"]
    / on_time_delivery["total_deliveries"]
    * 100
).round(2)

on_time_delivery

,total_deliveries,on_time_deliveries,on_time_rate
0,96470,88644,91.89


In [101]:
print(f"On-time Delivery Rate: {on_time_delivery.loc[0, 'on_time_rate']:.2f}%")

On-time Delivery Rate: 91.89%


### Business Insight

배송 완료 주문 기준 정시 배송률은 91.89%로 집계되었다.

전체 배송의 약 92%가 예상 배송일 이전(또는 동일한 날짜)에 완료되었으며, 약 7,800건의 주문은 예상 배송일 이후에 배송되었다.

정시 배송률은 전반적으로 높은 수준을 보였지만, 지연 주문이 특정 지역이나 배송 환경에 집중되어 있는지는 추가 분석이 필요한 부분이다.

# 08. Customer Analysis

고객 수는 서비스 성장의 핵심 지표이지만, 단순 고객 수만으로는 고객 유지 성과를 파악하기 어렵다.

본 섹션에서는 재구매 고객 비율(Repeat Customer Rate)을 분석하여 고객 유지 현황을 확인한다.

## Repeat Customer Rate

동일한 `customer_unique_id`가 2회 이상 주문한 고객을 재구매 고객으로 정의한다.

전체 고객 대비 재구매 고객의 비율을 계산하여 고객 유지 현황을 확인한다.

In [105]:
repeat_customers = pd.read_sql("""
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_orders DESC;
""", conn)

repeat_customers.head(10)

,customer_unique_id,total_orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6


In [168]:
repeat_customer_count = len(repeat_customers)

print(f"Repeat Customers: {repeat_customer_count:,}")

Repeat Customers: 2,997


In [170]:
repeat_customer_rate = (
    repeat_customer_count
    / unique_customers.loc[0, "unique_customers"]
    * 100
)

print(f"Repeat Customer Rate: {repeat_customer_rate:.2f}%")

Repeat Customer Rate: 3.12%


### Business Insight

전체 고객 중 재구매 고객은 2,997명으로, 재구매율은 3.12%로 집계되었다.

대부분의 고객은 단일 주문에 그쳤으며, 반복 구매 고객의 비중은 상대적으로 낮게 나타났다.

이는 현재 주문이 신규 고객 중심으로 발생하고 있음을 보여주며, 재구매 고객을 확대할 수 있는 여지가 있음을 시사한다.